# 01 — Procesamiento de texto para detección de fraude

**Objetivo:** dado el `title` y la `description` de un anuncio, emitir un
veredicto de riesgo textual **binario** (`fraude` | `limpio`) con **`reasons`**
explicables, combinando reglas deterministas (Capa 0) con clasificación
zero-shot del LLM local Qwen3 (Capa 1).

La lógica vive en el módulo **`text_risk.py`** (fuente única, reutilizada por
`02_fusion_isolation_forest.ipynb` y borrador del futuro `TextRiskService`).
Este notebook la recorre etapa por etapa y la evalúa contra un set semilla.

## Estrategia
- **Etapa 0 — Normalización + des-ofuscación**: NFKC, minúsculas, números
  escritos a dígitos (`cinco cinco` → `55`), `arroba`→`@`, `punto com`→`.com`,
  colapso de separadores en teléfonos.
- **Etapa 1 — Reglas (Capa 0)**: teléfono MX, correo, URL (**bloqueo duro** →
  fraude directo); precio y urgencia/CTA (señales que se anexan como contexto).
- **Etapa 2 — LLM zero-shot binario (Capa 1)**: Qwen3 clasifica `fraude`|`limpio`
  con salida JSON restringida y ejemplos few-shot (incluye evasión sutil sin
  datos de contacto y marketing legítimo).
- **Etapa 3 — Fusión**: `TextRiskResult`. `is_fraud_text = regla_dura ∨ (label == "fraude")`.
  Sin umbral numérico que calibrar.

> **Kernel:** `notebooks/.venv` (Python 3.12 con `llama-cpp-python`).

## Configuración

In [ ]:
import sys, importlib
import text_risk as tr
importlib.reload(tr)

print("python:", sys.executable)          # debe apuntar a notebooks/.venv
print("backend:", tr.LLM_BACKEND, "| USE_LLM:", tr.USE_LLM)
print("modelo :", tr.MODEL_PATH)

# Para experimentar con un modelo más capaz (mejor en casos sutiles):
# tr.MODEL_PATH = str(tr._MODELS_DIR / "Qwen3-4B-Instruct-2507-Q4_K_M.gguf")
# Para trabajar solo con reglas (sin cargar el LLM):
# tr.USE_LLM = False


## Etapa 0 — Normalización y des-ofuscación

La pieza clave: los defraudadores ofuscan el contacto (`cinco cinco 14...`,
`juan arroba gmail punto com`, `55-14-56`). Normalizamos **antes** de extraer
para que las reglas no se evadan con trucos de redacción.

In [ ]:
ej = "Llama al cinco cinco 14-56-78-90 o a juan ARROBA gmail punto com"
base  = tr.normalizar(ej)
desof = tr.desofuscar(base)
print("original :", ej)
print("normaliz.:", base)
print("desofusc.:", desof)
print("telefono :", tr.colapsar_telefono(desof))


## Etapa 1 — Capa 0: extractores deterministas

Cada extractor devuelve sus coincidencias (para la explicabilidad).
`has_hard_contact` = teléfono ∨ correo ∨ URL → **bloqueo duro** (fraude sin
consultar al LLM). Precio y urgencia se anexan como contexto pero **no** deciden
por sí solos.

In [ ]:
sig = tr.evaluate_rules("Departamento en renta",
                        "Llama al 55 1456 7890 de inmediato, oportunidad única")
print(sig)
print("bloqueo duro:", sig.has_hard_contact)


## Etapa 2 — Capa 1: Qwen3 como clasificador zero-shot binario

Un solo prompt fusiona *Information Extractor* + *Fraud Intent Detector* y
decide `fraude`|`limpio`. La salida se restringe con
`response_format={"type":"json_object","schema":...}` (en `llama-cpp-python` el
esquema se compila a gramática GBNF, así que el modelo **solo puede** emitir ese
JSON). Ejemplos few-shot en el prompt enseñan al modelo la evasión sutil y el
marketing legítimo. Si el modelo no está disponible, `classify_llm` devuelve
`None` y el pipeline degrada a solo reglas.

In [ ]:
# Caso sutil: sin datos de contacto, intención de evasión. Lo decide el LLM.
if tr.USE_LLM:
    print(tr.classify_llm(
        "Departamento céntrico",
        "No preguntes por aquí, hablemos directo y te hago mejor precio por fuera"))


## Etapa 3 — Fusión → `TextRiskResult`

- **Contacto directo (regla dura)** → `fraude`, sin llamar al LLM.
- **Si no** → decide el LLM (`label`). `is_fraud_text = label == "fraude"`.
- Sin LLM disponible → conservador: solo precio **y** urgencia juntos disparan fraude.
- `reasons` = reglas ∪ razones del LLM (dedupe).

In [ ]:
for t, d in [
    ("Compra ya a 500 pesos", "Aparta hoy mismo, última oportunidad"),
    ("Bonita casa en Polanco", "3 recámaras, 2 baños, cocina integral y jardín."),
]:
    print(tr.evaluate_text_risk(t, d))


## Evaluación sobre un set semilla

Como no hay datos etiquetados, validamos con un conjunto armado a mano: casos
fraudulentos (incluyendo **ofuscados** y **evasión sutil sin contacto**) y
**legítimos** con lenguaje comercial (para medir falsos positivos). Este set es
también la semilla del futuro clasificador supervisado.

In [ ]:
import pandas as pd

CASOS = [
    # --- Fraude ---
    {"title": "Compra ya a 500 pesos", "description": "Aparta hoy mismo, última oportunidad", "esperado": "fraude"},
    {"title": "Departamento en renta", "description": "Llama al 55 1456 7890 de inmediato para una oferta", "esperado": "fraude"},
    {"title": "Casa económica", "description": "Escríbeme a juan arroba gmail punto com y cerramos", "esperado": "fraude"},
    {"title": "Oferta única", "description": "Contáctame por WhatsApp al +52 55 8899 0011", "esperado": "fraude"},
    {"title": "Casa amplia", "description": "whatsapp cinco cinco 14 56 78 90, no preguntes aquí", "esperado": "fraude"},
    {"title": "Renta directa", "description": "más info en mi pagina casasbaratas punto com", "esperado": "fraude"},
    {"title": "Departamento céntrico", "description": "No preguntes por aquí, hablemos directo y te hago mejor precio por fuera", "esperado": "fraude"},
    # --- Legítimos (no deben marcarse) ---
    {"title": "Bonita casa en Polanco", "description": "3 recámaras, 2 baños, cocina integral, jardín y estacionamiento para 2 autos.", "esperado": "limpio"},
    {"title": "Departamento luminoso", "description": "80 m2, excelente ubicación cerca del metro, ideal para pareja.", "esperado": "limpio"},
    {"title": "Loft moderno", "description": "Espacio abierto con acabados de lujo, terraza y mucha luz natural.", "esperado": "limpio"},
    {"title": "Casa en condominio", "description": "Fraccionamiento con seguridad 24/7, alberca y áreas verdes, oportunidad única.", "esperado": "limpio"},
    {"title": "Casa remodelada", "description": "Amplia sala, comedor y 3 recámaras. Excelente oportunidad de inversión.", "esperado": "limpio"},
]

rows = []
for c in CASOS:
    res = tr.evaluate_text_risk(c["title"], c["description"])
    rows.append({
        "titulo": c["title"][:30],
        "esperado": c["esperado"],
        "obtenido": res.label,
        "ok": "✅" if res.label == c["esperado"] else "❌",
        "fuente": res.source,
        "reasons": " | ".join(res.reasons),
    })

df = pd.DataFrame(rows)
aciertos = (df["esperado"] == df["obtenido"]).sum()
print(f"Aciertos: {aciertos}/{len(df)}")
pd.set_option("display.max_colwidth", 80)
df


### Notas

- Con `tr.USE_LLM = False` la evaluación es **solo reglas**: atrapa el contacto
  (incluido ofuscado) pero deja pasar la evasión sutil sin datos de contacto —
  esa es tarea del LLM.
- Clasificación **binaria** (`fraude`|`limpio`): sin umbral que calibrar; la
  decisión es directa y explicable por `reasons`.
- Siguiente paso: `02_fusion_isolation_forest.ipynb` — acoplar este veredicto
  con el score del Isolation Forest en el Decision Engine final.